# VERA v2 — M3 detector-box paper campaign

This notebook reads the fresh M2 detector arrays from the attached `vera-v2-detector-outputs` Kaggle dataset, copies labels into writable storage, verifies the BioViL-T feature contract, and runs the faithful detector-box main row first. Remaining paper ablations are opt-in.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, zipfile
REPO_URL = 'https://github.com/hiennguyendang/phase_2_3_4_5.git'
def bootstrap_repo():
    target=Path('/kaggle/working/vera_repo')
    if (target/'.git').exists():
        try:
            subprocess.run(['git','-C',str(target),'fetch','origin','main'],check=True,stdout=subprocess.DEVNULL)
            subprocess.run(['git','-C',str(target),'reset','--hard','origin/main'],check=True,stdout=subprocess.DEVNULL)
            print('source: synced GitHub commit',subprocess.check_output(['git','-C',str(target),'rev-parse','HEAD'],text=True).strip()); return target
        except Exception as exc: print('[refresh] existing clone unavailable:',exc)
    if (target/'phase_2/scripts/yolo/5-infer_yolo.py').exists(): return target
    if target.exists(): shutil.rmtree(target)
    try:
        subprocess.run(['git','clone',REPO_URL,str(target)],check=True)
        print('source: GitHub commit',subprocess.check_output(['git','-C',str(target),'rev-parse','HEAD'],text=True).strip()); return target
    except Exception as exc: print('[fallback] GitHub clone unavailable:',exc)
    root=Path('/kaggle/input/datasets/nguynnghin/vera-v2-code'); archives=list(root.glob('*.zip')) if root.exists() else []; candidates=[root]
    if len(archives)==1:
        extracted=Path('/kaggle/working/vera_v2_code_extracted'); extracted.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archives[0]) as zf: zf.extractall(extracted)
        candidates=[extracted]+list(extracted.glob('*/'))
    for candidate in candidates:
        if (candidate/'phase_2/scripts/yolo/5-infer_yolo.py').exists(): shutil.copytree(candidate,target,dirs_exist_ok=True); print('source: Kaggle code dataset',candidate); return target
    raise RuntimeError('GitHub unavailable; attach /kaggle/input/datasets/nguynnghin/vera-v2-code')
REPO_DIR=bootstrap_repo()
sys.path.insert(0, str(REPO_DIR/'kaggle_notebooks'))
from vera_common import find_bundle, find_m2_outputs, FEATURE_ROOT, configure_drive, copy_tree
bundle = find_bundle()
m2_labels = find_m2_outputs()
remote = configure_drive()
print('bundle:', bundle, 'detector labels:', m2_labels, 'features:', FEATURE_ROOT)

In [ ]:
# Build a writable detector-label directory; never mutate /kaggle/input/datasets/nguynnghin.
labels = Path('/kaggle/working/m3_labels_detector_v2')
if labels.exists(): shutil.rmtree(labels)
copy_tree(bundle/'m3_labels_base', labels)
m2 = m2_labels
for name in ['boxes_det.npy','present_mask_det.npy','detector_provenance.json']:
    src=m2/name
    if not src.exists(): raise FileNotFoundError(src)
    shutil.copy2(src, labels/name)
# Launcher preflight expects the data-level concept-space copy.
(REPO_DIR/'data').mkdir(exist_ok=True)
shutil.copy2(labels/'m3_concept_space.json', REPO_DIR/'data/m3_concept_space.json')
provenance=json.loads((labels/'detector_provenance.json').read_text())
print('labels:', labels)
print('detector provenance:', provenance['detector_checkpoint_sha256'])

In [ ]:
# Feature smoke test and manifest coverage. Scan the flat 252k-file cache once.
import torch, numpy as np
feature_files = [Path(e.path) for e in os.scandir(FEATURE_ROOT) if e.is_file() and Path(e.name).suffix in {'.pt','.npy'}]
if not feature_files: feature_files = [p for p in FEATURE_ROOT.rglob('*') if p.suffix in {'.pt','.npy'}]
assert feature_files, FEATURE_ROOT
sample = feature_files[0]
x = torch.load(sample, map_location='cpu') if sample.suffix == '.pt' else np.load(sample, mmap_mode='r')
if isinstance(x, dict): x = next(v for v in x.values() if torch.is_tensor(v))
print('feature files:', len(feature_files), 'sample:', sample.name, tuple(x.shape), getattr(x,'dtype',None))
assert tuple(x.shape) in {(197,512),(196,512)}
rows = [json.loads(s) for s in (labels/'manifest.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
ids = {str(r['image_id']) for r in rows}
feature_ids = {p.stem for p in feature_files}
missing=ids-feature_ids
print('manifest IDs:',len(ids),'missing features:',len(missing))
assert not missing, f'feature cache does not cover manifest; first={sorted(missing)[:5]}'
assert tuple(np.load(labels/'boxes_det.npy',mmap_mode='r').shape)==(len(rows),29,4)
assert tuple(np.load(labels/'present_mask_det.npy',mmap_mode='r').shape)==(len(rows),29)
assert int(provenance.get('manifest_rows',len(rows)))==len(rows), 'M2 output was aligned to a different manifest length'

In [ ]:
# Shared launcher configuration. Conservative settings for each T4.
env = os.environ.copy()
env.update(PY='python', DEVICE='cuda:0', BATCH='8', W='2', EVAL_W='2', EP='40', CAL_BOOTSTRAP='200',
           FEAT=str(FEATURE_ROOT), LABELS=str(labels), RUNS='/kaggle/working/m3_runs',
           LOGDIR='/kaggle/working/m3_logs', DIAGDIR='/kaggle/working/m3_diagnostics',
           SYNC_REMOTE=remote+'/m3_runs', SYNC_DIAG_REMOTE=remote+'/m3_diagnostics', SYNC_EVERY='300')
subprocess.run(['rclone','copy',remote+'/m3_diagnostics','/kaggle/working/m3_diagnostics'],check=False)
subprocess.run(['bash',str(REPO_DIR/'phase_3/run_paper_m3_v2.sh'),'--profile','local4060','--scope','preflight'],cwd=REPO_DIR,env=env,check=True)
# Main faithful detector-box row first.
subprocess.run(['bash',str(REPO_DIR/'phase_3/run_paper_m3_v2.sh'),'--profile','local4060','--scope','main'],cwd=REPO_DIR,env=env,check=True)
subprocess.run(['rclone','copy','/kaggle/working/m3_runs/m3v2_vera_graph_lse_det',remote+'/m3_runs/m3v2_vera_graph_lse_det'],check=True)
subprocess.run(['rclone','copy','/kaggle/working/m3_diagnostics',remote+'/m3_diagnostics'],check=True)
cal=Path('/kaggle/working/m3_diagnostics/m3v2_vera_graph_lse_det.calibration.SUCCESS.json')
assert cal.exists(), 'M3 validation calibration did not complete'
print('main M3 checkpoint + pair thresholds + concept gate saved to Drive:',cal)

In [ ]:
# Optional: run the remaining retained rows two at a time, one process per T4.
RUN_REMAINING = False  # change to True only after the main row above finishes and is accepted
names = ['m3v2_no_concept_det','m3v2_concept_mlp_det','m3v2_graph_global_fusion_det',
         'm3v2_global_only_det','m3v2_graph_attention_det','m3v2_graph_mean_det',
         'm3v2_graph_max_det','m3v2_vera_graph_lse_gt']
if not RUN_REMAINING:
    print('remaining ablations skipped; set RUN_REMAINING=True when ready')
for start in range(0, len(names), 2) if RUN_REMAINING else []:
    procs=[]
    for gpu, name in enumerate(names[start:start+2]):
        e=env.copy(); e.update(DEVICE=f'cuda:{gpu}', RUN_NAME=name)
        procs.append(subprocess.Popen(['bash',str(REPO_DIR/'phase_3/run_paper_m3_v2.sh'),'--profile','local4060','--scope','all'],cwd=REPO_DIR,env=e))
    for p in procs:
        assert p.wait()==0
if RUN_REMAINING:
    subprocess.run(['rclone','copy','/kaggle/working/m3_runs',remote+'/m3_runs'],check=True)
    subprocess.run(['rclone','copy','/kaggle/working/m3_diagnostics',remote+'/m3_diagnostics'],check=True)
    print('all retained M3 rows and diagnostics saved to Drive')